In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# --- Download data ---
ticker = "NVDA"  # or AMD
data = yf.download(ticker, start="2018-01-01", end="2025-01-01")

# Fix columns if needed
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

df = data[['Close']].copy()

# --- Compute RSI (14) ---
delta = df['Close'].diff()

gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()

rs = gain / (loss + 1e-9)
df['RSI'] = 100 - (100 / (1 + rs))

# --- Normalize RSI ---
df['RSI_norm'] = (df['RSI'] - 50) / 50

# --- IMPORTANT: shift to avoid lookahead ---
df['RSI_norm'] = df['RSI_norm'].shift(1)

# --- Direction prediction ---
# If RSI > 50 → bullish, else bearish
df['Pred_Dir'] = np.where(df['RSI_norm'] > 0, 1, -1)

# --- Actual direction ---
df['Actual_Dir'] = np.sign(df['Close'].diff())

df = df.dropna()

# --- Train/Test split ---
split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

# --- Accuracy ---
acc = (test['Pred_Dir'] == test['Actual_Dir']).mean()

# --- Naive baseline ---
test['Naive'] = test['Close'].shift(1)
test = test.dropna()

naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

# --- Results ---
print("════════════════════════════════════")
print("RSI-ONLY MODEL")
print("──────────────")
print(f"RSI Accuracy     : {acc:.2%}")
print(f"Naive Accuracy   : {naive_acc:.2%}")
print(f"Improvement      : {(acc - naive_acc):+.2%}")
print("════════════════════════════════════")

[*********************100%***********************]  1 of 1 completed

════════════════════════════════════
RSI-ONLY MODEL
──────────────
RSI Accuracy     : 51.43%
Naive Accuracy   : 49.28%
Improvement      : +2.14%
════════════════════════════════════


In [2]:
df = data.copy()

ma_window = 20
momentum_window = 5
alpha = 0.15

# --- Base features ---
df['MA'] = df['Close'].rolling(ma_window).mean()
df['Residual'] = df['Close'] - df['MA']
df['Residual_Pred'] = df['Residual'].shift(1)
df['Momentum'] = (df['Close'] - df['Close'].shift(momentum_window)).shift(1)

# --- RSI ---
delta = df['Close'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df['RSI'] = 100 - (100 / (1 + rs))
df['RSI_norm'] = ((df['RSI'] - 50) / 50).shift(1)

# --- Baseline prediction ---
df['Pred_Base'] = df['MA'] + df['Residual_Pred'] + alpha * df['Momentum']

# --- RSI Filtered Prediction ---
# Only apply momentum when RSI agrees with direction
df['Momentum_Filtered'] = np.where(
    np.sign(df['Momentum']) == np.sign(df['RSI_norm']),
    df['Momentum'],
    0
)

df['Pred_RSI_Filter'] = (
    df['MA'] 
    + df['Residual_Pred'] 
    + alpha * df['Momentum_Filtered']
)

# --- Directions ---
df['Actual'] = np.sign(df['Close'].diff())
df['Dir_Base'] = np.sign(df['Pred_Base'].diff())
df['Dir_Filter'] = np.sign(df['Pred_RSI_Filter'].diff())

df = df.dropna()

split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

base_acc = (test['Actual'] == test['Dir_Base']).mean()
filter_acc = (test['Actual'] == test['Dir_Filter']).mean()

# Naive
test['Naive'] = test['Close'].shift(1)
test = test.dropna()
naive_acc = (np.sign(test['Close'].diff()) == np.sign(test['Naive'].diff())).mean()

print("════════════════════════════════════")
print(f"Naive Accuracy        : {naive_acc:.2%}")
print("──────────────")
print(f"Baseline Accuracy     : {base_acc:.2%}")
print(f"RSI-Filtered Accuracy : {filter_acc:.2%}")
print("──────────────")
print(f"Change vs Baseline    : {(filter_acc - base_acc):+.2%}")
print("════════════════════════════════════")

════════════════════════════════════
Naive Accuracy        : 49.28%
──────────────
Baseline Accuracy     : 52.59%
RSI-Filtered Accuracy : 51.44%
──────────────
Change vs Baseline    : -1.15%
════════════════════════════════════


In [3]:
import numpy as np
import pandas as pd

df = data.copy()

# --- Parameters (same baseline) ---
ma_window = 20
momentum_window = 5
alpha = 0.15
cost = 0.0005  # transaction cost (0.05%)

# --- Features ---
df['MA'] = df['Close'].rolling(ma_window).mean()
df['Residual'] = df['Close'] - df['MA']
df['Residual_Pred'] = df['Residual'].shift(1)
df['Momentum'] = (df['Close'] - df['Close'].shift(momentum_window)).shift(1)

# --- Prediction ---
df['Prediction'] = df['MA'] + df['Residual_Pred'] + alpha * df['Momentum']

# --- Signal (IMPORTANT: shift to avoid lookahead) ---
df['Signal'] = np.where(df['Prediction'] > df['Close'], 1, -1)
df['Signal'] = df['Signal'].shift(1)

# --- Returns ---
df['Return'] = df['Close'].pct_change()
df['Strategy_Return'] = df['Signal'] * df['Return']

# --- Transaction cost (only when position changes) ---
df['Trade'] = df['Signal'].diff().abs()
df['Strategy_Return'] -= df['Trade'] * cost

df = df.dropna()

# --- Train/Test split ---
split = int(len(df) * 0.8)
test = df.iloc[split:].copy()

# --- Performance ---
test['Cumulative_Market'] = (1 + test['Return']).cumprod()
test['Cumulative_Strategy'] = (1 + test['Strategy_Return']).cumprod()

# --- Metrics ---
sharpe = np.sqrt(252) * test['Strategy_Return'].mean() / (test['Strategy_Return'].std() + 1e-9)

# Max Drawdown
cum = test['Cumulative_Strategy']
rolling_max = cum.cummax()
drawdown = (cum - rolling_max) / rolling_max
max_dd = drawdown.min()

# --- Results ---
print("════════════════════════════════════")
print("BACKTEST RESULTS")
print("──────────────")
print(f"Final Strategy Return : {test['Cumulative_Strategy'].iloc[-1]:.2f}x")
print(f"Final Market Return   : {test['Cumulative_Market'].iloc[-1]:.2f}x")
print("──────────────")
print(f"Sharpe Ratio          : {sharpe:.2f}")
print(f"Max Drawdown          : {max_dd:.2%}")
print("════════════════════════════════════")

════════════════════════════════════
BACKTEST RESULTS
──────────────
Final Strategy Return : 0.69x
Final Market Return   : 3.29x
──────────────
Sharpe Ratio          : -0.32
Max Drawdown          : -45.24%
════════════════════════════════════
